In [ ]:
import json
from pathlib import Path

json_path = Path("/home/yuhaowang/project/report_generation/TRRG/R2GenGPT/merged_iuxray.json")

threshold = 0.5           # 过滤阈值
splits = ["train", "val", "test"]

# 读取
with json_path.open("r", encoding="utf-8") as f:
    data = json.load(f)

# 遍历并生成 report
for split in splits:
    for sample in data.get(split, []):
        prompts = []
        for view in sample.get("views", []):
            for triple in view.get("triples", []):
                if triple.get("p_finding", 0) > threshold:
                    bp = triple.get("best_prompt", "").strip()
                    if bp and bp not in prompts:      # 去重
                        prompts.append(bp)

        # 写入 report（若没有符合条件的条目则写空串）
        sample["report"] = ", ".join(prompts)

# 保存
with json_path.open("w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print("Done! 所有样本已补充 report 并保存。")


In [ ]:
import json
knowledge_path = '/home/yuhaowang/project/report_generation/TRRG/R2GenGPT/merged_mimic.json'
json_path = '/data2/yuhaowang/MIMIC-CXR/mimic_annotation_all.json'
with open(json_path, 'r') as f:
    json_data = json.load(f)
with open(knowledge_path, 'r') as f:
    knowledge_data = json.load(f)
#print(json_data['test'][0])
knowledge_data['test'][0]

In [ ]:
import json
knowledge_path = '/home/yuhaowang/project/report_generation/TRRG/R2GenGPT/merged_iuxray.json'
json_path = '/data2/yuhaowang/MIMIC-CXR/mimic_annotation_all.json'
with open(json_path, 'r') as f:
    json_data = json.load(f)
with open(knowledge_path, 'r') as f:
    knowledge_data = json.load(f)



{'id': 'CXR3030_IM-1405',
 'num_views': 2,
 'views': [{'image': '/data2/yuhaowang/iu_xray/images/CXR3030_IM-1405/0.png',
   'topk_findings': [{'name': 'Support Devices', 'p': 0.8628131151199341},
    {'name': 'No Finding', 'p': 0.8097341656684875},
    {'name': 'Pleural Other', 'p': 0.5715629458427429},
    {'name': 'Edema', 'p': 0.38147708773612976}],
   'triples': [{'finding': 'Support Devices',
     'p_finding': 0.8628131151199341,
     'best_prompt': 'chest tube right IJ approach',
     'severity': None,
     'subtype': 'chest tube',
     'location': 'right IJ approach'},
    {'finding': 'No Finding',
     'p_finding': 0.8097341656684875,
     'best_prompt': 'lungs are clear',
     'severity': None,
     'subtype': 'lungs are clear',
     'location': None},
    {'finding': 'Pleural Other',
     'p_finding': 0.5715629458427429,
     'best_prompt': 'empyema (pleural collection) right',
     'severity': None,
     'subtype': 'empyema (pleural collection)',
     'location': 'right'},
 

In [ ]:
# for split, items in knowledge_data.items():
#     for item in items:
#         # 取出 report；若不存在或为空，使用默认文本
#         report_text = item.pop("report", "").strip()
#         if not report_text:
#             report_text = "there is no findings"
#         item["knowledge"] = report_text


# with open(knowledge_path, 'w') as f:
#     json.dump(knowledge_data, f)

In [10]:
import json
knowledge_path = '/home/yuhaowang/project/report_generation/TRRG/R2GenGPT/merged_iuxray.json'

with open(knowledge_path, 'r') as f:
    knowledge_data = json.load(f)

knowledge_data['test'][1]['knowledge']

'there is no findings'

In [13]:
import json
import os

knowledge_path = '/home/yuhaowang/project/report_generation/TRRG/R2GenGPT/merged_iuxray.json'
json_path = '/data2/yuhaowang/iu_xray/annotation.json'

# 读取
with open(json_path, 'r', encoding='utf-8') as f:
    json_data = json.load(f)
with open(knowledge_path, 'r', encoding='utf-8') as f:
    knowledge_data = json.load(f)

# 1) 先把 json_data 里的 {id -> report} 建索引（包含 train/val/test）
report_map = {}
for split in ('train', 'val', 'test'):
    for item in json_data.get(split, []):
        rid = item.get('id')
        if rid is None:
            continue
        # 只接受确实存在 report 字段的样本
        if 'report' in item and isinstance(item['report'], str):
            report_map[rid] = item['report']

# 2) 写回到 knowledge_data（严格覆盖为来自 json_data 的 report）
stats = {'total_items': 0, 'updated': 0, 'missing_in_json_data': 0}
for split in ('train', 'val', 'test'):
    items = knowledge_data.get(split, [])
    for it in items:
        stats['total_items'] += 1
        rid = it.get('id')
        if rid in report_map:
            it['report'] = report_map[rid]   # 覆盖/写入
            stats['updated'] += 1
        else:
            stats['missing_in_json_data'] += 1

# 3) 保存（默认另存为 *_with_reports.json；如需覆盖原文件，把 out_path 改为 knowledge_path）
root, ext = os.path.splitext(knowledge_path)
out_path = root + '_with_reports.json'

with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(knowledge_data, f, ensure_ascii=False, indent=2)

print('Done.')
print('Output:', out_path)
print('Stats:', stats)


Done.
Output: /home/yuhaowang/project/report_generation/TRRG/R2GenGPT/merged_iuxray_with_reports.json
Stats: {'total_items': 2955, 'updated': 2955, 'missing_in_json_data': 0}


In [16]:
test_path = '/home/yuhaowang/project/report_generation/TRRG/R2GenGPT/merged_iuxray_with_reports.json'
with open(test_path, 'r', encoding='utf-8') as f:
    json_data = json.load(f)
json_data['test'][0]


{'id': 'CXR3030_IM-1405',
 'num_views': 2,
 'views': [{'image': '/data2/yuhaowang/iu_xray/images/CXR3030_IM-1405/0.png',
   'topk_findings': [{'name': 'Support Devices', 'p': 0.8628131151199341},
    {'name': 'No Finding', 'p': 0.8097341656684875},
    {'name': 'Pleural Other', 'p': 0.5715629458427429},
    {'name': 'Edema', 'p': 0.38147708773612976}],
   'triples': [{'finding': 'Support Devices',
     'p_finding': 0.8628131151199341,
     'best_prompt': 'chest tube right IJ approach',
     'severity': None,
     'subtype': 'chest tube',
     'location': 'right IJ approach'},
    {'finding': 'No Finding',
     'p_finding': 0.8097341656684875,
     'best_prompt': 'lungs are clear',
     'severity': None,
     'subtype': 'lungs are clear',
     'location': None},
    {'finding': 'Pleural Other',
     'p_finding': 0.5715629458427429,
     'best_prompt': 'empyema (pleural collection) right',
     'severity': None,
     'subtype': 'empyema (pleural collection)',
     'location': 'right'},
 